# Module 07 · Unit 02
# Adversarial Enumeration

| | |
|---|---|
| **Estimated time** | 60–70 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Adversarial Enumeration \[AE\] · Attack Graphs \[AG\] |
| **Refresher** | Module 07 · Unit 01 — Counting Principles |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Compute the exact size of an **LLM token vocabulary space** for sequences of length $k$
2. Formalise an **adversarial example budget** as a combination count in a discrete feature space
3. Enumerate **attack paths** in a graph and compute the total path count between two vertices
4. Define the **feasibility threshold** — the count above which exhaustive search is impractical
5. Apply the **counting hierarchy** to compare the difficulty of different attack classes
6. Connect adversarial enumeration to the formal security property framework of Module 02


## 🔣 Symbol Primer

This unit introduces one new notation and reuses the counting symbols from Unit 01.

| Symbol | Name | Read it as | Meaning |
|---|---|---|---|
| $\Sigma$ | Alphabet | "sigma" | The set of symbols an LLM can generate (vocabulary) |
| $\Sigma^k$ | Sequences of length $k$ | "sigma to the $k$" | All strings of exactly $k$ tokens |
| $\Sigma^{\leq k}$ | Sequences up to length $k$ | "sigma up to $k$" | $\Sigma^0 \cup \Sigma^1 \cup \cdots \cup \Sigma^k$ |
| $\epsilon$ | Empty string | "epsilon" | The string of length zero |
| $B_\epsilon(x)$ | Ball of radius $\epsilon$ | "epsilon-ball around $x$" | All inputs within distance $\epsilon$ of $x$ |

> **The connection to Module 09.** $\Sigma^*$ — all strings of any length over $\Sigma$
> — is the formal language studied in Module 09 (Automata). Here we count its finite
> subsets; there we specify which strings a system accepts.

---


## 1 · The LLM Token Vocabulary Space

A large language model operates on **tokens** — subword units that partition text
into chunks. A typical model (GPT-4 class) uses a vocabulary of approximately
$|\Sigma| = 100{,}000$ tokens. A sequence of $k$ tokens is an element of $\Sigma^k$.

### Counting Sequences

By the multiplication rule, the number of distinct $k$-token sequences is:

$$|\Sigma^k| = |\Sigma|^k$$

For $|\Sigma| = 100{,}000$:

| $k$ (tokens) | Approximate words | $|\Sigma^k|$ | $\log_2$ |
|:---:|:---:|:---:|:---:|
| 1 | ~0.75 words | $10^5$ | 17 |
| 10 | ~7.5 words | $10^{50}$ | 166 |
| 50 | ~37 words | $10^{250}$ | 830 |
| 100 | ~75 words | $10^{500}$ | 1,660 |

These numbers are astronomical — exhaustive enumeration of all 100-token prompts
is incomparably harder than brute-forcing an AES-256 key ($2^{256} \approx 10^{77}$).

### What This Means for Jailbreak Search

An **automated jailbreak search** that tries to find a specific token sequence
$x \in \Sigma^k$ causing unwanted model behaviour is not an exhaustive enumeration
problem — the space is too large. Instead, jailbreak attacks use:

1. **Greedy token substitution** — modify one token at a time, keeping changes that
   increase a proxy objective. The search space at each step is $|\Sigma| = 100{,}000$
   — tractable.

2. **Gradient-based search** — use the model's gradient to identify which token
   substitutions increase the probability of the target output. This is the GCG
   (Greedy Coordinate Gradient) attack.

3. **Human-in-the-loop** — a human red-teamer writes prompts; far fewer than $|\Sigma^k|$
   total attempts, but each is informed by understanding of the model.

The key insight: **the vastness of $\Sigma^k$ does not protect the model**. Structured
search through a tiny fraction of the space can still find effective adversarial prompts.
The security guarantee is not "the search space is large" — it is "the set of effective
adversarial prompts is too sparse to find without the gradient."


## 2 · Adversarial Example Budgets in Discrete Feature Spaces

For a machine learning classifier operating on **discrete Boolean features**
(like the access control classifier from Module 06), we can count adversarial
examples precisely.

### The $\epsilon$-Neighbourhood

Given an input $x = (x_1, x_2, \ldots, x_n) \in \{0,1\}^n$ and a budget
$\epsilon$ (maximum number of features we can flip), the **adversarial budget
set** is:

$$B_\epsilon(x) = \{x' \in \{0,1\}^n \mid d_H(x, x') \leq \epsilon\}$$

where $d_H(x, x')$ is the **Hamming distance** — the number of coordinates in
which $x$ and $x'$ differ.

$$|B_\epsilon(x)| = \sum_{k=0}^{\epsilon} \binom{n}{k}$$

This counts all inputs reachable from $x$ by flipping at most $\epsilon$ features:
$\binom{n}{0}$ inputs at distance 0 (just $x$ itself), $\binom{n}{1}$ at distance 1,
and so on.

### Security Interpretation

For the four-attribute ABAC classifier $(n=4, A, T, S, M)$ with current input
$x = (1,1,0,0)$ (admin, in-window, not-suspended, no-MFA):

$$|B_1(x)| = \binom{4}{0} + \binom{4}{1} = 1 + 4 = 5$$
$$|B_2(x)| = \binom{4}{0} + \binom{4}{1} + \binom{4}{2} = 1 + 4 + 6 = 11$$

With a budget of flipping 1 feature, there are 4 candidate adversarial inputs.
With a budget of 2, there are 10 (plus the original). The attacker who can
influence two attributes has a much larger neighbourhood to search.

For a real ML security classifier with $n = 50$ features and $\epsilon = 3$:

$$|B_3(x)| = \binom{50}{0} + \binom{50}{1} + \binom{50}{2} + \binom{50}{3}
= 1 + 50 + 1225 + 19600 = 20{,}876$$

Only 20,876 candidates — completely enumerable by any modern computer.
The adversarial robustness problem at small $\epsilon$ is not a counting problem;
it is a question of which inputs in $B_\epsilon(x)$ achieve the target label.


## 3 · Attack Path Enumeration

In Module 05 we found the **minimum-cost** attack path using Dijkstra. A complementary
question: how many distinct attack paths exist between two vertices? This measures
the attacker's flexibility and the defender's coverage requirement.

### Simple Path Count

The number of simple paths (no vertex repeated) between $s$ and $t$ in a graph
can vary enormously depending on graph structure. For a **layered graph** with
$L$ layers of $k$ vertices each, where every vertex in layer $i$ connects to
every vertex in layer $i+1$, the number of simple $s$-to-$t$ paths is:

$$k^{L-1}$$

Each of the $k$ intermediate layers offers $k$ choices independently.

### Security Interpretation

More paths = more attacker flexibility = harder to block. From Module 05 Unit 03,
the **minimum edge cut** $\lambda(G, s, t)$ is a lower bound on how many paths
can be simultaneously blocked. By Menger's theorem:

$$\lambda(G, s, t) = \text{number of edge-disjoint paths from } s \text{ to } t$$

So the minimum cut equals the number of completely independent attack routes —
routes that share no edge. This is the intersection of counting (how many paths)
and graph algorithms (which edges to cut).

**Counting paths vs cutting paths:** knowing the total path count tells you the
attacker's search space. Knowing the edge-disjoint path count tells you the
defender's minimum investment to block all routes.


## 4 · The Feasibility Threshold

Not all large numbers are equal from a security perspective. The **feasibility
threshold** separates counts that are attackable from those that are not, given
realistic attacker resources.

### A Practical Scale

| Count | Time at $10^9$/sec | Feasibility |
|---|---|---|
| $2^{20} \approx 10^6$ | 1 ms | Instant — no security |
| $2^{30} \approx 10^9$ | 1 sec | Trivially feasible |
| $2^{40} \approx 10^{12}$ | 17 min | Feasible for any attacker |
| $2^{56}$ (DES key space) | ~1 year | Feasible for well-funded attackers |
| $2^{64} \approx 10^{19}$ | 585 years (single machine) | Borderline — GPU clusters feasible |
| $2^{80}$ | $\sim 10^{13}$ years | Infeasible for any current technology |
| $2^{128}$ (AES-128) | $\approx 10^{29}$ years | Computationally infeasible |

The threshold is not a hard line — it depends on the attacker's resources, time
horizon, and whether parallel computation is available. As a rule of thumb:

$$\text{Secure count} \geq 2^{128} \quad \text{for any random secret}$$
$$\text{Safe birthday bound} \geq 2^{64} \quad \text{for any token/nonce/hash output}$$

### Connecting Counting to the Module 02 Security Property

A formal security property from Module 02:

$$\forall a \in \text{Attackers},\; \text{cost}(\text{find}(a, \text{secret})) > \text{budget}(a)$$

The counting framework quantifies $\text{cost}$: if the search space has $N$ elements
and the attacker must evaluate each, the cost is $O(N)$ operations. Security holds
when $N$ exceeds the attacker's computational budget.

This is the formal connection between combinatorics (Module 07) and security
specifications (Module 02) — counting is how you turn a qualitative guarantee
("the secret is hard to find") into a quantitative one ("finding the secret
requires $2^{128}$ operations").


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[AE\] \[AG\]

| Enumeration concept | Security meaning | Countermeasure |
|---|---|---|
| $|\Sigma^k|$ | LLM jailbreak search space | Gradient-based search bypasses exhaustion |
| $|B_\epsilon(x)|$ | Adversarial example budget | Adversarial training covers $B_\epsilon$ for all training points |
| Simple path count | Attacker's routing flexibility | More paths = more edge cuts needed |
| Edge-disjoint paths | Independent attack routes | Min-cut is the minimum defence investment |
| $2^{80}$ threshold | Minimum secure secret size | Keys, tokens, nonces all need $\geq 2^{128}$ space |
| Birthday bound $\sqrt{N}$ | Collision feasibility | Nonces/tokens need $2 \times$ more bits than keys |

**The adversarial enumeration principle.** Security does not come from large
search spaces alone — it comes from the combination of large spaces *and* the
absence of shortcuts. An AES key space of $2^{128}$ is secure not because
$2^{128}$ is large (though it is), but because no known attack reduces the
effective search below $2^{128}$. An LLM prompt space of $10^{500}$ is not
necessarily secure, because gradient-based attacks provide an enormous shortcut.

Counting tells you the size of the space. Security analysis tells you whether
shortcuts exist. Both are necessary.

---


## Python: Visualization & Verification

**1 · Token Space Calculator** — compute and visualise the LLM token sequence
space for realistic vocabulary and sequence lengths; compare to cryptographic
key spaces to establish scale.

**2 · Adversarial Budget Explorer** — compute $|B_\epsilon(x)|$ for the ABAC
classifier across all inputs and budgets; identify which inputs are most vulnerable
(have the most adversarial neighbours in the ALLOW class).

**3 · Attack Path Counter** — enumerate all simple paths between entry and target
in the enterprise attack graph; count total paths, edge-disjoint paths, and
visualise the relationship between path count and min-cut.


In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)
print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from math import comb, log2, log10, factorial, sqrt
import itertools

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')


### 1 · Token Space Calculator

We compute the size of the token sequence space for a range of vocabulary sizes
and sequence lengths, then compare these numbers to cryptographic key spaces
and feasibility thresholds on a unified log scale.


In [ ]:
# ── 1 · Token Space Calculator ───────────────────────────────────────────────

vocab_sizes = {
    'GPT-2 (50k tokens)':      50_000,
    'GPT-4 class (100k tokens)': 100_000,
    'Character-level (256)':    256,
    'ASCII printable (94)':     94,
}

sequence_lengths = [1, 5, 10, 20, 50, 100]

print("Token Space Sizes (log₂) for Various Vocabulary and Sequence Lengths")
print(f"{'Model/Vocab':<30} " + "  ".join(f"k={k:>3}" for k in sequence_lengths))
print("─" * 85)

space_data = {}
for name, vocab in vocab_sizes.items():
    row = []
    for k in sequence_lengths:
        log_size = k * log2(vocab)
        row.append(log_size)
    space_data[name] = row
    print(f"  {name:<28} " + "  ".join(f"{v:>6.0f}" for v in row))

# Key reference points
print()
print("Reference points (log₂):")
refs = [
    ("DES key space",        56),
    ("AES-128 key space",   128),
    ("AES-256 key space",   256),
    ("SHA-256 output",      256),
    ("Feasibility threshold", 80),
]
for label, bits in refs:
    print(f"  {label:<30} {bits}")

# ── Visualisation: sequence length vs log space size ──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

ax = axes[0]
colors_v = [TS_BLUE, TS_AMBER, TS_GREEN, TS_RED]
k_range  = range(1, 51)

for (name, vocab), color in zip(vocab_sizes.items(), colors_v):
    log_sizes = [k * log2(vocab) for k in k_range]
    ax.plot(k_range, log_sizes, color=color, lw=2.5, label=name)

# Reference lines
for label, bits, color in [
    ('AES-128 (2¹²⁸)', 128, TS_GREEN),
    ('Feasible threshold (2⁸⁰)', 80, TS_AMBER),
    ('DES (2⁵⁶)', 56, TS_RED),
]:
    ax.axhline(bits, color=color, lw=1.5, linestyle='--', alpha=0.6, label=label)

ax.set_xlabel('Sequence length $k$ (tokens)')
ax.set_ylabel('$\log_2(|\Sigma^k|)$  — bits of search space')
ax.set_title('Token Sequence Space vs Length', fontweight='bold', pad=8)
ax.legend(fontsize=8, ncol=2)

# Right: comparison table as a heatmap
ax2 = axes[1]
matrix = np.array(space_data[list(vocab_sizes.keys())[1]])  # GPT-4 class
ax2.barh(range(len(sequence_lengths)),
         [k * log2(100_000) for k in sequence_lengths],
         color=[TS_RED if k * log2(100_000) < 80 else
                TS_AMBER if k * log2(100_000) < 128 else TS_GREEN
                for k in sequence_lengths],
         edgecolor='white', height=0.65)

for i, k in enumerate(sequence_lengths):
    log_s = k * log2(100_000)
    label = f"k={k}: 10^{log10(100_000**k):.0f} combinations"
    ax2.text(log_s + 5, i, label, va='center', fontsize=8, color=TS_GRAY)

ax2.axvline(80,  color=TS_AMBER, lw=1.5, linestyle='--', label='Feasibility threshold (2⁸⁰)')
ax2.axvline(128, color=TS_GREEN, lw=1.5, linestyle='--', label='AES-128 threshold (2¹²⁸)')
ax2.set_yticks(range(len(sequence_lengths)))
ax2.set_yticklabels([f'k = {k}' for k in sequence_lengths])
ax2.set_xlabel('$\log_2$ of token sequence space')
ax2.set_title('GPT-4 Class (100k vocab)
Red=easy, Amber=borderline, Green=hard',
              fontweight='bold', pad=8)
ax2.legend(fontsize=8)

fig.patch.set_facecolor('white')
plt.suptitle('LLM Token Sequence Space — Scale Comparison',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nKey insight: even a 5-token prompt from a 100k vocabulary has")
print(f"  10^{5*log10(100_000):.0f} possible strings — vastly beyond brute force.")
print("  Jailbreak attacks work NOT by exhaustion but by structured search")
print("  (gradient descent, greedy substitution) through this enormous space.")


**What do you see?**

Even at sequence length $k = 5$, the GPT-4 vocabulary space ($10^{25}$) dwarfs
AES-128. At $k = 10$ it is incomparably larger than any cryptographic key space.
The colour coding makes the feasibility threshold visual: short sequences (red)
are technically enumerable; longer sequences (green) are not.

The right-hand bar chart crystallises the point: every sequence length used in
real prompts sits firmly in the "hard" (green) zone. Exhaustive jailbreak search
is impossible — and yet jailbreaks are found. The lesson is that search-space
size is not the same as security. Gradient-based attacks operate in a completely
different regime, finding effective adversarial prompts in thousands of queries
against a space of $10^{250}$.


### 2 · Adversarial Budget Explorer — ABAC Classifier

We compute $|B_\epsilon(x)|$ for every input to the four-attribute ABAC classifier
and identify the inputs that are most exposed: those with the most ALLOW-labelled
neighbours within a small Hamming ball.


In [ ]:
# ── 2 · Adversarial Budget Explorer ─────────────────────────────────────────

# Four-attribute ABAC: A=admin, T=time, S=suspended, M=MFA
features   = ['A', 'T', 'S', 'M']
n_features = len(features)

# Correct policy: A ∧ T ∧ ¬S ∧ M
def classify(x):
    A, T, S, M = x
    return 'ALLOW' if (A and T and not S and M) else 'DENY'

all_inputs = list(itertools.product([0, 1], repeat=n_features))
labels     = {x: classify(x) for x in all_inputs}

def hamming_distance(x, y):
    return sum(a != b for a, b in zip(x, y))

def adversarial_neighbourhood(x, epsilon, target_label='ALLOW'):
    """Return count of inputs within Hamming distance epsilon with target_label."""
    return [y for y in all_inputs
            if hamming_distance(x, y) <= epsilon and labels[y] == target_label]

# Compute adversarial exposure for each input at epsilon=1,2,3
print("Adversarial Exposure Analysis — ABAC Classifier (A ∧ T ∧ ¬S ∧ M)")
print(f"{'Input (A,T,S,M)':<20} {'Label':>6} "
      + "  ".join(f"ε={e} ALLOW nbrs" for e in [1,2,3]))
print("─" * 72)

exposure_data = []
for x in all_inputs:
    lbl = labels[x]
    row = [x, lbl]
    for eps in [1, 2, 3]:
        nbrs = adversarial_neighbourhood(x, eps, 'ALLOW')
        row.append(len(nbrs))
    exposure_data.append(row)
    flag = '  ← ALLOW (target)' if lbl == 'ALLOW' else ''
    print(f"  {str(x):<18}  {lbl:>6}  "
          + "  ".join(f"{row[2+i]:>13}" for i in range(3))
          + flag)

# ── Exposure heatmap ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
# Bar chart: inputs sorted by ε=2 ALLOW neighbours
sorted_data = sorted(exposure_data, key=lambda r: -r[3])  # sort by ε=2
input_labels = [str(r[0]) for r in sorted_data]
eps2_counts  = [r[3] for r in sorted_data]
bar_colors   = [TS_GREEN if r[1] == 'ALLOW' else TS_BLUE for r in sorted_data]

bars = ax.barh(range(len(input_labels)), eps2_counts, color=bar_colors,
               edgecolor='white', height=0.7)
ax.set_yticks(range(len(input_labels)))
ax.set_yticklabels(input_labels, fontsize=8, fontfamily='monospace')
ax.set_xlabel('ALLOW-labelled inputs within ε=2 Hamming ball')
ax.set_title('Adversarial Exposure per Input (ε=2)
'
             'Green = already ALLOW; Blue = DENY inputs',
             fontweight='bold', pad=8)

legend = [mpatches.Patch(color=TS_GREEN, label='ALLOW inputs'),
          mpatches.Patch(color=TS_BLUE,  label='DENY inputs')]
ax.legend(handles=legend, fontsize=9)

# Right: neighbourhood size formula verification
ax2 = axes[1]
epsilons = range(0, n_features + 1)
theoretical = [sum(comb(n_features, k) for k in range(e+1)) for e in epsilons]

ax2.plot(epsilons, theoretical, color=TS_BLUE, lw=2.5, marker='o',
         markersize=9, label=r'$\sum_{k=0}^{\epsilon}inom{n}{k}$, $n=4$')
ax2.fill_between(epsilons, theoretical, alpha=0.15, color=TS_BLUE)

for e, t in zip(epsilons, theoretical):
    ax2.text(e, t + 0.3, str(t), ha='center', fontsize=11, fontweight='bold')

ax2.set_xlabel('Budget ε (max features flipped)')
ax2.set_ylabel('$|B_\epsilon(x)|$ — neighbourhood size')
ax2.set_title(r'Adversarial Budget $|B_\epsilon(x)|$ for $n=4$ Features',
              fontweight='bold', pad=8)
ax2.set_xticks(epsilons)
ax2.legend(fontsize=9)

fig.patch.set_facecolor('white')
plt.suptitle('ABAC Adversarial Budget Analysis',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\nSecurity insight:")
print("  A DENY input with many ALLOW neighbours in B_ε is highly exposed.")
print("  An attacker who can flip ε features can reach ALLOW from those inputs.")
allow_x = next(x for x in all_inputs if labels[x] == 'ALLOW')
print(f"  The ALLOW input {allow_x} itself has {adversarial_neighbourhood(allow_x, 1, 'DENY').__len__()} DENY neighbours at ε=1.")
print(f"  → Flipping any ONE feature from the ALLOW input causes a DENY.")


**What do you see?**

The exposure bar chart reveals which DENY inputs are most at risk of adversarial
manipulation: inputs with many ALLOW-labelled neighbours at $\epsilon = 2$.
An attacker who can modify two attribute values on those inputs can reach the
ALLOW leaf.

The neighbourhood size formula confirms the combinatorial counts: at $\epsilon = 4$
(all features can be flipped), the neighbourhood covers all 16 inputs — the entire
input space. At $\epsilon = 0$ it covers only the input itself.

The security insight at the bottom is important: the single ALLOW input
$(A=1, T=1, S=0, M=1)$ has four DENY neighbours at $\epsilon = 1$ — flipping
any one feature causes a denial. This means the ALLOW state is fragile: any
single attribute failure denies access. This is the **principle of least privilege**
expressed as adversarial robustness: the ALLOW region is small and surrounded by DENY.


### 3 · Attack Path Enumeration in the Enterprise Network

We count all simple paths between the internet entry point and the domain
controller in the enterprise attack graph. We then compare the total path count
to the number of edge-disjoint paths — the minimum cut — to show the relationship
between attacker flexibility and defender investment.


In [ ]:
# ── 3 · Attack Path Enumeration ──────────────────────────────────────────────

AG = nx.DiGraph()
AG.add_edges_from([
    ('internet','web_server'), ('internet','api_server'),
    ('web_server','load_balancer'), ('web_server','api_server'),
    ('api_server','db_primary'), ('api_server','log_server'),
    ('api_server','dev_laptop'), ('load_balancer','db_primary'),
    ('db_primary','db_replica'), ('db_primary','backup_server'),
    ('db_primary','admin_workstation'), ('dev_laptop','admin_workstation'),
    ('dev_laptop','log_server'), ('admin_workstation','domain_controller'),
    ('admin_workstation','log_server'), ('log_server','backup_server'),
])

src, tgt = 'internet', 'domain_controller'

# Count all simple paths
all_paths = list(nx.all_simple_paths(AG, src, tgt))
min_cut   = nx.minimum_edge_cut(AG, src, tgt)
edge_dj   = list(nx.edge_disjoint_paths(AG, src, tgt))

print(f"Attack Path Enumeration: {src} → {tgt}")
print(f"  Total simple paths:          {len(all_paths)}")
print(f"  Shortest path length:        {min(len(p)-1 for p in all_paths)} hops")
print(f"  Longest path length:         {max(len(p)-1 for p in all_paths)} hops")
print(f"  Edge-disjoint paths:         {len(edge_dj)}")
print(f"  Minimum edge cut:            {len(min_cut)} edges")
print(f"  Cut edges:                   {sorted(min_cut)}")
print()
print("All simple paths (sorted by length):")
for i, path in enumerate(sorted(all_paths, key=len), 1):
    print(f"  {i:>2}. ({len(path)-1} hops) {' → '.join(path)}")

# ── Visualisation ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: path length distribution
ax = axes[0]
lengths = [len(p) - 1 for p in all_paths]
unique_lengths = sorted(set(lengths))
counts_by_len  = [lengths.count(l) for l in unique_lengths]

ax.bar(unique_lengths, counts_by_len, color=TS_BLUE, edgecolor='white', width=0.6)
for length, count in zip(unique_lengths, counts_by_len):
    ax.text(length, count + 0.05, str(count), ha='center',
            fontsize=11, fontweight='bold', color=TS_BLUE)
ax.set_xlabel('Path length (hops)')
ax.set_ylabel('Number of distinct paths')
ax.set_title(f'Attack Path Length Distribution
{src} → {tgt}',
             fontweight='bold', pad=8)
ax.set_xticks(unique_lengths)

# Right: attack graph with all paths highlighted
ax2 = axes[1]
pos = nx.spring_layout(AG, seed=42, k=2.5)

# Draw all edges faintly
nx.draw_networkx_edges(AG, pos, ax=ax2, edge_color='#dddddd',
                       width=1, arrows=True, arrowsize=12,
                       connectionstyle='arc3,rad=0.06')

# Colour edges by how many paths use them
edge_usage = {}
for path in all_paths:
    for u, v in zip(path, path[1:]):
        edge_usage[(u,v)] = edge_usage.get((u,v), 0) + 1

max_usage = max(edge_usage.values())
for (u, v), usage in edge_usage.items():
    intensity = usage / max_usage
    color = plt.cm.YlOrRd(0.3 + 0.7 * intensity)
    nx.draw_networkx_edges(AG, pos, ax=ax2, edgelist=[(u,v)],
                           edge_color=[color], width=1 + 3*intensity,
                           arrows=True, arrowsize=16,
                           connectionstyle='arc3,rad=0.06')

nc = [TS_GREEN if v == src else
      TS_RED   if v == tgt else TS_BLUE
      for v in AG.nodes()]
nx.draw_networkx_nodes(AG, pos, ax=ax2, node_color=nc, node_size=800, alpha=0.92)
nx.draw_networkx_labels(AG, pos, ax=ax2, font_size=7,
                        font_color='white', font_weight='bold')
ax2.set_title(f'Edge Usage Heatmap ({len(all_paths)} total paths)
'
              f'Darker = used by more paths — higher priority to monitor',
              fontweight='bold', pad=8)
ax2.axis('off'); ax2.set_facecolor('white')

fig.patch.set_facecolor('white')
plt.suptitle('Attack Path Enumeration: Count, Length, and Edge Usage',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nDefender insight:")
print(f"  {len(all_paths)} total paths exist — the attacker has {len(all_paths)} possible routes.")
print(f"  Only {len(min_cut)} edge cut(s) needed to block ALL paths (Menger's theorem).")
print(f"  The cut edges are the highest-priority monitoring and hardening targets.")
print(f"  Edge usage heatmap identifies edges appearing in the most paths — monitor these first.")


**What do you see?**

The path length distribution shows that most paths are 3–5 hops — the attacker
has substantial flexibility in how they route through the network. The minimum
path is 3 hops; there are longer alternatives if the short paths are monitored.

The edge usage heatmap is the actionable output: darker edges appear in more
attack paths. An edge used by every path is a **mandatory traversal** — monitoring
or cutting it affects all attack routes. An edge used by only one path is less
critical but still represents a unique attack route.

The key ratio: $n$ total paths, minimum cut of $k$ edges. This directly answers
the security team's question: *"how many monitoring rules do I need to catch all
attacks?"* — at minimum, $k$ rules covering the cut edges. More rules covering
high-usage edges provide defence in depth.

**The Module 05–07 connection.** Module 05 found the cheapest path (Dijkstra);
Module 07 counts all paths (combinations) and identifies the minimum cut
(edge-disjoint paths = Menger's theorem). Together they answer: *which path
would the attacker take, how many alternatives do they have, and what is the
minimum investment to block them all?*


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. ADVERSARIAL ROBUSTNESS CERTIFICATE
#    For the multi-label ABAC classifier from Module 06 Unit 03
#    (with labels ALLOW, DENY, ESCALATE, LOG_AND_ALLOW, QUARANTINE):
#    For each non-ALLOW input, compute: what is the minimum ε such that
#    an ALLOW-labelled neighbour exists in B_ε(x)?
#    This is the "adversarial distance to ALLOW" — a robustness certificate.
#    Which inputs are hardest to adversarially push to ALLOW?
#    Which are easiest?
#
# 2. LLM JAILBREAK SEARCH SPACE
#    Model a simplified greedy token substitution attack:
#      - Start with a benign prompt of 10 tokens
#      - At each step, substitute ONE token with any of |Σ|=1000 alternatives
#      - After k steps, how many distinct prompts have been explored?
#    Compare: exhaustive (1000^10), greedy (1000 × k), random beam search (beam × k).
#    Plot all three on a log scale for k from 1 to 20.
#    What does this tell you about why jailbreaks are found despite the huge space?
#
# 3. COUNTING PATHS WITH CONSTRAINTS
#    In the attack graph, some paths are only feasible for certain attacker profiles:
#      Profile A (external): can only start at 'internet'
#      Profile B (insider):  can only start at 'dev_laptop'
#    For each profile, count:
#      (a) Total simple paths to domain_controller
#      (b) Minimum cut (independent attack routes)
#      (c) Paths with length ≤ 3 hops (quick attacks)
#    Which profile is more dangerous, and why?

# Your work here:


---

## Summary

| Concept | Formula | Security application |
|---|---|---|
| $|\Sigma^k|$ | $|\Sigma|^k$ | LLM token sequence space — exhaustion infeasible |
| $|B_\epsilon(x)|$ | $\sum_{i=0}^{\epsilon}\binom{n}{i}$ | Adversarial example budget in discrete space |
| Simple path count | Enumerated via DFS | Attacker routing flexibility |
| Edge-disjoint paths | = Min edge cut (Menger) | Minimum defence investment |
| Feasibility threshold | $\approx 2^{80}$ | Below this, exhaustive search is practical |
| Birthday bound | $\approx \sqrt{N}$ | Collision threshold for tokens/nonces |

---

## Module 07 Complete

| Unit | Content | Payoff |
|---|---|---|
| **01** | Multiplication, addition, inclusion-exclusion, permutations, combinations, birthday bound | The exact size of any attack surface, key space, or search space |
| **02** | LLM token spaces, adversarial budgets, attack path enumeration, feasibility thresholds | Counting applied directly to AI security threat modeling |

---

## Up Next

**Module 08 — Number Theory & Modular Arithmetic**

We shift from counting to structure within numbers. Number theory provides the
mathematical foundations of every cryptographic primitive securing AI deployment:
TLS connections to APIs, key exchange between agents, digital signatures on model
weights. RSA, Diffie-Hellman, and AES all rest on number-theoretic facts that
are provable with the tools from this course.

$\rightarrow$ `modules/module-08/unit-01-divisibility-modular-arithmetic.ipynb`
